# Inferred EEG networks — all patients, one grid per patient

Scans a folder of `ABC_Results_chbXX_<rec>_<period>` subfolders, discovers every patient automatically, and saves **one network grid per patient** to the `figures/` folder as a PNG. Nothing is drawn inline.

Populations 1–4 = channels **FP1-F7, FP1-F3, FP2-F4, FP2-F8**. Solid dark arrow = edge present (posterior mode 1); dotted orange = uncertain (posterior mean in [1/3, 2/3]).

## 0. Unzip the results

Extracts `All_Results.zip` (next to this notebook) if not already done. Handles both layouts: a wrapping `All_Results/` folder, or the `ABC_Results_*` folders at the zip's top level. Sets `ROOT` accordingly.


In [1]:
import os, zipfile

ZIP = "All_Results.zip"
ROOT = "All_Results"

def _has_runs(path):
    return os.path.isdir(path) and any(
        n.startswith("ABC_Results_") for n in os.listdir(path))

if not _has_runs(ROOT):
    if not os.path.exists(ZIP):
        raise FileNotFoundError(
            f"{ZIP} not found in {os.getcwd()!r}. "
            f"Put the notebook next to the zip, or fix ZIP/ROOT.")
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(".")
    if not _has_runs(ROOT) and any(
            n.startswith("ABC_Results_") for n in os.listdir(".")):
        ROOT = "."
    print(f"extracted {ZIP}; ROOT = {ROOT!r}")
else:
    print(f"already extracted; ROOT = {ROOT!r}")

assert _has_runs(ROOT), f"no ABC_Results_* folders found under {ROOT!r}"
print(sum(n.startswith("ABC_Results_") for n in os.listdir(ROOT)), "run folders visible")

already extracted; ROOT = 'All_Results'
274 run folders visible


## 1. Setup

In [2]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # no inline display; we only save PNGs
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle

# ROOT is set by the unzip cell above
OUTDIR = "figures"
os.makedirs(OUTDIR, exist_ok=True)

N = 4
CHANNELS = ["FP1-F7", "FP1-F3", "FP2-F4", "FP2-F8"]
PAIRS = [(j, k) for j in range(1, N + 1) for k in range(1, N + 1) if j != k]
UNCERTAIN = (1/3, 2/3)         # posterior-mean band drawn dotted/orange


## 2. Discover patients and load particles

Folder names look like `ABC_Results_chb15_06_before`. The patient is the `chbXX` token; the record is `chbXX_06`; the period is `before`/`during`.

In [3]:
FOLDER_RE = re.compile(r"ABC_Results_(chb\d+)_(.+)_(before|during)$")


def load_run(folder):
    """Return {(j,k): particle array} for one ABC_Results_* folder."""
    out = {}
    for j, k in PAIRS:
        path = os.path.join(folder, f"p{j}{k}vec.txt")
        if os.path.exists(path):
            out[(j, k)] = np.loadtxt(path).ravel()
    return out


# runs[patient][(record, period)] = particles
runs = {}
skipped = []
for name in sorted(os.listdir(ROOT)):
    folder = os.path.join(ROOT, name)
    if not os.path.isdir(folder):
        continue
    m = FOLDER_RE.match(name)
    if not m:
        skipped.append(name)
        continue
    patient, record, period = m.group(1), f"{m.group(1)}_{m.group(2)}", m.group(3)
    particles = load_run(folder)
    if particles:
        runs.setdefault(patient, {})[(record, period)] = particles

patients = sorted(runs)
print(f"{len(patients)} patients found: {patients}")
for p in patients:
    recs = sorted({r for r, _ in runs[p]})
    print(f"  {p}: {len(recs)} records, {len(runs[p])} run folders")
if skipped:
    print(f"\n{len(skipped)} folders skipped (name did not match): {skipped[:5]}"
          + (" ..." if len(skipped) > 5 else ""))

23 patients found: ['chb01', 'chb02', 'chb03', 'chb04', 'chb05', 'chb06', 'chb07', 'chb08', 'chb09', 'chb10', 'chb11', 'chb12', 'chb13', 'chb14', 'chb15', 'chb16', 'chb18', 'chb19', 'chb20', 'chb21', 'chb22', 'chb23', 'chb24']
  chb01: 7 records, 14 run folders
  chb02: 2 records, 4 run folders
  chb03: 7 records, 14 run folders
  chb04: 3 records, 6 run folders
  chb05: 5 records, 10 run folders
  chb06: 6 records, 12 run folders
  chb07: 3 records, 6 run folders
  chb08: 5 records, 10 run folders
  chb09: 3 records, 6 run folders
  chb10: 6 records, 12 run folders
  chb11: 3 records, 5 run folders
  chb12: 10 records, 20 run folders
  chb13: 8 records, 16 run folders
  chb14: 7 records, 14 run folders
  chb15: 14 records, 28 run folders
  chb16: 6 records, 12 run folders
  chb18: 6 records, 12 run folders
  chb19: 3 records, 6 run folders
  chb20: 6 records, 12 run folders
  chb21: 4 records, 8 run folders
  chb22: 3 records, 6 run folders
  chb23: 3 records, 6 run folders
  chb24: 1

## 3. Posterior modes and means

For each binary parameter, the posterior **mean** is the fraction of particles equal to 1 (probability the edge exists); the **mode** is 1 when that exceeds 0.5 (the edge drawn).

In [4]:
def summarise(particles):
    means = {pair: float(v.mean()) for pair, v in particles.items()}
    modes = {pair: int(m > 0.5) for pair, m in means.items()}
    return modes, means


summaries = {p: {key: summarise(pt) for key, pt in runs[p].items()}
             for p in patients}

## 4. Network drawing

In [5]:
NODE_XY = {i: (i - 1.0, 0.0) for i in range(1, N + 1)}
NODE_R = 0.12


def strength_label(j, k):
    d = abs(j - k)
    return "L" if d == 1 else ("cL" if d == 2 else f"c$^{d-1}$L")


def draw_network(modes, means, ax, title="", show_labels=True):
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_xlim(-0.55, N - 0.45)
    ax.set_ylim(-1.35, 1.35)

    for j, k in PAIRS:
        if not modes.get((j, k), 0):
            continue
        m = means[(j, k)]
        uncertain = UNCERTAIN[0] <= m <= UNCERTAIN[1]

        d = abs(j - k)
        rad = 0.22 * d

        x1, y1 = NODE_XY[j]
        x2, y2 = NODE_XY[k]
        arrow = FancyArrowPatch(
            (x1, y1), (x2, y2),
            connectionstyle=f"arc3,rad={rad}",
            arrowstyle="-|>", mutation_scale=11,
            linewidth=1.1 if not uncertain else 1.0,
            linestyle=(0, (2, 2)) if uncertain else "solid",
            color="tab:orange" if uncertain else "0.15",
            shrinkA=9, shrinkB=9,
            zorder=1,
        )
        ax.add_patch(arrow)

        if show_labels:
            xm, ym = (x1 + x2) / 2, (y1 + y2) / 2
            apex = rad / 2 * abs(x2 - x1)
            ax.text(xm, ym + apex * 1.12, strength_label(j, k),
                    ha="center", va="center", fontsize=7,
                    color="tab:orange" if uncertain else "0.35", zorder=3)

    for i, (x, y) in NODE_XY.items():
        ax.add_patch(Circle((x, y), NODE_R, facecolor="white",
                            edgecolor="black", linewidth=1.1, zorder=2))
        ax.text(x, y, str(i), ha="center", va="center", fontsize=10, zorder=3)
        ax.text(x, y - NODE_R - 0.16, CHANNELS[i - 1], ha="center", va="top",
                fontsize=7, color="0.4", zorder=3)

    if title:
        ax.set_title(title, fontsize=10)

## 5. Save one grid per patient

Rows = recordings; left column = before, right = during. Each patient's grid is written to `figures/<patient>_networks_all.png`. No figures are shown inline.

In [6]:
def grid_for_patient(patient):
    s = summaries[patient]
    recordings = sorted({rec for rec, _ in s})
    nrow = len(recordings)
    fig, axes = plt.subplots(nrow, 2, figsize=(9, 2.9 * nrow), squeeze=False)

    for r, rec in enumerate(recordings):
        for c, period in enumerate(["before", "during"]):
            ax = axes[r][c]
            key = (rec, period)
            if key in s:
                modes, means = s[key]
                draw_network(modes, means, ax, title=f"{rec} — {period}")
            else:
                ax.axis("off")
                ax.set_title(f"{rec} — {period} (missing)",
                             fontsize=9, color="0.6")

    fig.suptitle(f"{patient}:  solid = edge present (mode 1);  "
                 f"dotted orange = mean in [1/3, 2/3]",
                 fontsize=9, color="0.4", y=1.001)
    fig.tight_layout()
    out = os.path.join(OUTDIR, f"{patient}_networks_all.png")
    fig.savefig(out, dpi=200, bbox_inches="tight")
    plt.close(fig)          # free memory; nothing shown inline
    return out


saved = [grid_for_patient(p) for p in patients]
print(f"Saved {len(saved)} patient grids to {OUTDIR}/:")
for path in saved:
    print("  " + path)

Saved 23 patient grids to figures/:
  figures\chb01_networks_all.png
  figures\chb02_networks_all.png
  figures\chb03_networks_all.png
  figures\chb04_networks_all.png
  figures\chb05_networks_all.png
  figures\chb06_networks_all.png
  figures\chb07_networks_all.png
  figures\chb08_networks_all.png
  figures\chb09_networks_all.png
  figures\chb10_networks_all.png
  figures\chb11_networks_all.png
  figures\chb12_networks_all.png
  figures\chb13_networks_all.png
  figures\chb14_networks_all.png
  figures\chb15_networks_all.png
  figures\chb16_networks_all.png
  figures\chb18_networks_all.png
  figures\chb19_networks_all.png
  figures\chb20_networks_all.png
  figures\chb21_networks_all.png
  figures\chb22_networks_all.png
  figures\chb23_networks_all.png
  figures\chb24_networks_all.png


## 6. Save one CSV per patient

One file per patient in the `tables/` folder: `tables/<patient>_rho.csv`. Rows are recording × period (before/during); each coupling parameter contributes a `rho_jk_mode` (0/1) and a `rho_jk_mean` (posterior probability) column.


In [7]:
TABLEDIR = "tables"
os.makedirs(TABLEDIR, exist_ok=True)

def table_for_patient(patient):
    s = summaries[patient]
    rows = []
    for (record, period) in sorted(s):
        modes, means = s[(record, period)]
        row = {"recording": record, "period": period}
        for j, k in PAIRS:
            row[f"rho_{j}{k}_mode"] = modes.get((j, k), "")
            row[f"rho_{j}{k}_mean"] = round(means[(j, k)], 4) if (j, k) in means else ""
        rows.append(row)
    df = pd.DataFrame(rows)
    out = os.path.join(TABLEDIR, f"{patient}_rho.csv")
    df.to_csv(out, index=False)
    return out

saved_tables = [table_for_patient(p) for p in patients]
print(f"Saved {len(saved_tables)} patient CSVs to {TABLEDIR}/:")
for path in saved_tables:
    print("  " + path)

Saved 23 patient CSVs to tables/:
  tables\chb01_rho.csv
  tables\chb02_rho.csv
  tables\chb03_rho.csv
  tables\chb04_rho.csv
  tables\chb05_rho.csv
  tables\chb06_rho.csv
  tables\chb07_rho.csv
  tables\chb08_rho.csv
  tables\chb09_rho.csv
  tables\chb10_rho.csv
  tables\chb11_rho.csv
  tables\chb12_rho.csv
  tables\chb13_rho.csv
  tables\chb14_rho.csv
  tables\chb15_rho.csv
  tables\chb16_rho.csv
  tables\chb18_rho.csv
  tables\chb19_rho.csv
  tables\chb20_rho.csv
  tables\chb21_rho.csv
  tables\chb22_rho.csv
  tables\chb23_rho.csv
  tables\chb24_rho.csv


In [2]:
import glob, os
import pandas as pd

TABLEDIR = "tables"

# Combine all per-patient rho CSVs into one long table
paths = sorted(glob.glob(os.path.join(TABLEDIR, "*_rho.csv")))

frames = []
for path in paths:
    patient = os.path.basename(path).replace("_rho.csv", "")  # e.g. "chb01"
    df = pd.read_csv(path)
    df.insert(0, "patient", patient)
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)

out = os.path.join(TABLEDIR, "all_patients_rho.csv")
combined.to_csv(out, index=False)
print(f"Combined {len(paths)} files -> {out}")
print(f"Shape: {combined.shape[0]} rows x {combined.shape[1]} cols")
combined.head()

Combined 23 files -> tables\all_patients_rho.csv
Shape: 263 rows x 27 cols


,patient,recording,period,rho_12_mode,rho_12_mean,rho_13_mode,rho_13_mean,rho_14_mode,rho_14_mean,rho_21_mode,...,rho_32_mode,rho_32_mean,rho_34_mode,rho_34_mean,rho_41_mode,rho_41_mean,rho_42_mode,rho_42_mean,rho_43_mode,rho_43_mean
0,chb01,chb01_03,before,1,0.826,1,0.738,1,0.876,0,...,1,0.826,0,0.070,1,0.926,1,0.586,1,0.604
1,chb01,chb01_03,during,1,1.000,1,0.506,1,1.000,1,...,1,0.998,0,0.004,1,1.000,0,0.024,0,0.016
2,chb01,chb01_04,before,0,0.438,0,0.388,1,0.870,1,...,0,0.348,1,0.954,1,0.920,1,0.804,1,0.930
3,chb01,chb01_04,during,1,1.000,0,0.014,1,0.946,1,...,0,0.078,0,0.000,1,1.000,0,0.006,0,0.000
4,chb01,chb01_15,before,1,0.848,1,0.852,1,0.794,0,...,1,0.916,1,0.822,1,0.960,1,0.638,1,0.972
